In [ ]:
import os, sys
import numpy as np

# ---------------- constants ----------------
N        = 16000          # agents
STEPS    = 400            # time steps
DT       = 1e-2           # step size (Euler)|
DIM      = 1              # opinion is 1D; keep for symmetry with other codes
OUTDIR   = "data"

# interaction kernel: bounded-confidence + smooth weight
# EPS      = 0.35           # confidence radius ε
ELL      = 0.5           # smooth scale ℓ (in kernel exp(-(r/ℓ)^2))

# blocking to reduce memory
BI = 400
BJ = 400

# ---------------- init ----------------
def init_opinion(N, seed=0):
    rng = np.random.default_rng(seed)
    # bimodal-ish initial opinions for visible clustering pressure
    x = 0.5 * rng.standard_normal(N) + rng.choice([-0.8, +0.8], size=N, p=[0.5,0.5])
    # x = np.clip(x, -1.5, 1.5).astype(np.float64)
    return x, x.copy(), rng  # return current x, "anchor" x0 (for stubbornness), rng


# ---------------- kernel ----------------
def phi(r, eps=EPS, ell=ELL):
    # r: (Bi,Bj) opinion differences
    out = np.exp(-(r/ell)**2)
    return out

# ---------------- force (blocked) ----------------
def blocked_influence(x, bi=BI, bj=BJ):
    """
    Compute local, state-dependent influence:
      x_i' = x_i + DT * [  sum_j w_ij * (x_j - x_i)  /  sum_j w_ij  - s*(x_i - x0_i) ] + noise
    where w_ij = phi(x_j - x_i) * graph_ij (if A provided).
    Returns drift term g(x) of shape (N,).
    """
    N = x.shape[0]
    g = np.zeros_like(x)
    # sums for local normalization
    num = np.zeros_like(x)  # numerator sum_j w_ij * (x_j - x_i)
    den = np.zeros_like(x)  # denominator sum_j w_ij

    for ib in range(0, N, bi):
        i1 = min(ib + bi, N)
        xi = x[ib:i1][:, None]                   # (Bi,1)
        num_b = np.zeros((i1-ib,), dtype=x.dtype)
        den_b = np.zeros((i1-ib,), dtype=x.dtype)

        for jb in range(0, N, bj):
            j1 = min(jb + bj, N)
            xj = x[jb:j1][None, :]               # (1,Bj)

            diff = xj - xi                       # (Bi,Bj)
            w = phi(diff)                        # (Bi,Bj)

            # remove self when blocks overlap (i==j)
            if ib == jb:
                m = min(i1-ib, j1-jb)
                w[np.arange(m), np.arange(m)] = 0.0

            num_b += (w @ (x[jb:j1])) - (w.sum(axis=1) * x[ib:i1])
            den_b += w.sum(axis=1)

        num[ib:i1] = num_b
        den[ib:i1] = den_b + 1e-12  # avoid divide-by-zero

    # local normalized drift toward neighbors inside confidence
    g = num / den
    return g

# ---------------- step ----------------
def step(x, x0, rng, A=None):
    drift = blocked_influence(x, bi=BI, bj=BJ)
    x_next = x + DT * drift
    return x_next

# ---------------- main ----------------
def main():
    if len(sys.argv) < 1+1:
        print("Usage: python opinion_dynamics_local.py <seed:int>")
        sys.exit(1)
    seed = 1 #int(sys.argv[1])

    x, x0, rng = init_opinion(N, seed=seed)

    # save buffers
    X_save = np.empty((STEPS + 1, N), dtype=np.float64)
    X_save[0] = x

    # simple cluster diagnostic: empirical variance
    VAR_save = np.empty(STEPS + 1, dtype=np.float64)
    VAR_save[0] = x.var()

    for t in range(1, STEPS + 1):
        x = step(x, x0, rng)
        X_save[t] = x
        VAR_save[t] = x.var()

    os.makedirs(OUTDIR, exist_ok=True)
    np.save(os.path.join(OUTDIR, f"opinion_traj_{seed}.npy"), X_save)
    np.save(os.path.join(OUTDIR, f"opinion_var_{seed}.npy"), VAR_save)
    print(f"[OK] saved opinions -> {OUTDIR}/opinion_traj_{seed}.npy  shape={X_save.shape}")
    print(f"[OK] saved variance -> {OUTDIR}/opinion_var_{seed}.npy  shape={VAR_save.shape}")



In [ ]:
main()

In [ ]:
data = np.load("./data/opinion_traj_1.npy")
data.shape

In [ ]:
import matplotlib.pyplot as plt
plt.hist(data[0, :], bins=50, histtype="step", label="t=0")
plt.hist(data[-1, :], bins=50, histtype="step", label="t=end")
plt.legend()
plt.show()



In [ ]:
seed = 3
rng = np.random.default_rng(seed)
value = rng.integers(low=0, high=3, size=1)[0]
x = 0.5 * rng.standard_normal(N) + rng.choice([-value, +value], size=N, p=[0.5,0.5])
plt.hist(x, bins=50, histtype="step", label="t=0")


In [ ]:
rng.choice([-0.8, +0.8], size=N, p=[0.5,0.5])